# Урок № 7. Диалоговые окна в `tkinter` — создаём приложение «Заметки»

**Python Middle · Возраст: 10–15 лет · Практический Jupyter Notebook для преподавателя**

На прошлом занятии мы создавали окно авторизации. Теперь напишем небольшую настольную программу, в которой можно **создавать, открывать, редактировать и удалять заметки**.

> **Главная идея:** мы не собираем весь проект за один раз. Сначала учимся открывать диалоги, затем делаем окно редактора, затем учимся записывать текст в файл. В конце соединяем все части в приложение.

**Основа:** прикреплённая методичка «PYMiddle. Урок №7. Диалоговые окна в tkinter. Пишем приложение „Заметки“» (Материалы Journal). Все ключевые части и операции из неё включены в урок. В исходных фрагментах есть небольшие ошибки; в этом ноутбуке они исправлены, чтобы пример запускался.

## Как пользоваться ноутбуком

- **Текстовые ячейки (Markdown)** — объяснения и вопросы. Их читаем вместе с детьми.
- **Ячейки с кодом** — запускаем по очереди сочетанием `Shift + Enter`.
- **`%%writefile`** в начале ячейки означает «сохрани код ниже в настоящий `.py`-файл». Эту ячейку тоже запускаем через `Shift + Enter`.
- **Важное правило:** прежде чем запускать следующий этап, закрываем окна предыдущего. Приложения открываются **в отдельных окнах**, не внутри страницы Jupyter.
- Для работы нужен **локальный Jupyter Notebook или JupyterLab**, Python 3 и `tkinter`. В обычном Google Colab без собственного рабочего стола Tkinter-окна не появятся.

**Преподавателю:** демонстрируйте сначала ожидаемый результат («какое окно должно открыться»), потом запускайте код, затем показывайте 1–2 ключевые строки. Не требуйте от детей помнить все названия методов с первого раза.

## Маршрут занятия — ориентир на 160 минут

| Этап | Время | Результат |
|---|---:|---|
| Повторение и знакомство с диалогами | 20 мин | Окна с сообщениями, вопросами и вводом |
| `Toplevel`, `ScrolledText`, `grab_set` | 25 мин | Открываем редактор в отдельном окне |
| Класс `Note` и хранение заметок | 35 мин | Сохраняем/читаем/удаляем `.note` |
| `Methods`, список файлов | 15 мин | Ищем существующие заметки |
| Собираем интерфейс приложения | 35 мин | Карточки заметок, редактирование, удаление |
| Проверка, самостоятельные задания | 30 мин | Работающее приложение и мини-проект |

*Темп можно менять. Для детей 10–12 лет дольше разбирайте части 1–3, для 13–15 лет оставляйте больше самостоятельной практики.*

---
# Часть 0. Подготовка

Мы будем сохранять учебные `.py`-файлы в **отдельной папке** `tkinter_notes_lesson7`. В этой папке будет каталог `notes` с пользовательскими заметками — так программа не затронет другие файлы компьютера.

**Преподавателю:** запустите следующую служебную ячейку **один раз**. Позже для запуска программы будем вызывать функцию `run_gui("имя.py")`: она открывает новое окно без блокировки Jupyter.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tkinter as tk

PROJECT = Path.cwd() / "tkinter_notes_lesson7"
PROJECT.mkdir(exist_ok=True)
(PROJECT / "notes").mkdir(exist_ok=True)

open_apps = []

def run_gui(script_name="main.py"):
    """Запустить отдельную Tkinter-программу и не блокировать Jupyter."""
    script = PROJECT / script_name
    if not script.exists():
        print(f"Файл не найден: {script}")
        return None
    process = subprocess.Popen([sys.executable, "-u", script_name], cwd=PROJECT)
    open_apps.append(process)
    print(f"Открыто окно из файла {script_name}. Закройте его, когда закончите просмотр.")
    return process

print("Рабочая папка:", PROJECT)
print("Версия Tk:", tk.TkVersion)

### Мини-повторение прошлого урока

Спросите у класса:

1. Зачем нужна команда `tk.Tk()`?
2. Чем отличается `tk.Button` от `tk.Label`?
3. Что делает `command=...` у кнопки?
4. Чем `tk.Entry` отличается от большого текстового поля `tk.Text`?

**Ответы для преподавателя:** создаёт основное окно; кнопка реагирует на действия, надпись показывает текст; `command` назначает функцию при нажатии; `Entry` обычно одна строка, `Text` — несколько.

---
# Часть 1. Что такое диалоговое окно?

Диалог — это короткое «общение» между программой и человеком. Например:

- **Информация:** «Заметка сохранена!»
- **Предупреждение:** «Такое имя уже занято».
- **Ошибка:** «Некорректное имя файла».
- **Подтверждение:** «Точно удалить заметку?» — пользователь отвечает «Да» или «Нет».
- **Ввод:** «Как назвать новую заметку?» — пользователь печатает ответ.
- **Редактирование:** отдельное небольшое окно с текстом.

Библиотеки `tkinter.messagebox` и `tkinter.simpledialog` дают готовые диалоги, а `tk.Toplevel` позволяет создать своё окно.

**Вопрос:** почему программа не должна удалять файлы сразу, без вопроса? Потому что человек мог нажать кнопку случайно.

## 1. Первые сообщения: `showinfo`, `showwarning`, `showerror`

Создадим три кнопки для трёх видов сообщений. Всё, что находится после `command=`, выполняется только при нажатии кнопки.

> Закройте окно после изучения всех кнопок. Затем переходите к следующей ячейке.

In [ ]:
%%writefile tkinter_notes_lesson7/demo_01_messages.py
import tkinter as tk
from tkinter import messagebox

root = tk.Tk()
root.title("Три вида сообщений")
root.geometry("420x260")


def show_success():
    messagebox.showinfo("Информация", "Заметка сохранена!")


def show_warning():
    messagebox.showwarning("Внимание", "Сначала придумай название!")


def show_error():
    messagebox.showerror("Ошибка", "Не удалось выполнить действие")


tk.Label(root, text="Нажми на каждую кнопку", font=("Arial", 14)).pack(pady=12)
tk.Button(root, text="Информация", command=show_success).pack(pady=5)
tk.Button(root, text="Предупреждение", command=show_warning).pack(pady=5)
tk.Button(root, text="Ошибка", command=show_error).pack(pady=5)
root.mainloop()


In [ ]:
run_gui("demo_01_messages.py")

### Разбираем 3 самые важные строки

- `from tkinter import messagebox` — подключили готовые окошки.
- `messagebox.showinfo("Заголовок", "Текст")` — показываем сообщение, ничего не спрашивая.
- `tk.Button(..., command=show_success)` — передаём **имя функции без скобок**: функция вызовется при нажатии.

**Мини-задание 1:** поменяйте текст у каждой из трёх кнопок и сообщения. Заново запустите ячейку с `%%writefile` и снова откройте программу.

## 2. Диалоги, которые возвращают ответ

Важное отличие: диалог может не только показывать сообщение, но и **возвращать результат**.

- `messagebox.askyesno(...)` возвращает `True` при «Да» и `False` при «Нет».
- `simpledialog.askstring(...)` возвращает введённый текст, **или `None`, если нажали «Отмена»**.
- Пустой ответ `""` отличается от `None`: пустая строка означает, что пользователь нажал ОК, не введя название.

Здесь результат выводится прямо в интерфейс — детям не нужно искать его в терминале.

In [ ]:
%%writefile tkinter_notes_lesson7/demo_02_questions.py
import tkinter as tk
from tkinter import messagebox, simpledialog

root = tk.Tk()
root.title("Программа задаёт вопросы")
root.geometry("460x240")

answer_label = tk.Label(root, text="Ответ появится здесь", font=("Arial", 12))
answer_label.pack(pady=25)


def ask_delete():
    answer = messagebox.askyesno("Удаление", "Удалить учебную заметку?")
    if answer:
        answer_label.config(text="Ответ: ДА (True)")
    else:
        answer_label.config(text="Ответ: НЕТ (False)")


def ask_name():
    name = simpledialog.askstring("Новая заметка", "Как назовём заметку?")
    if name is None:
        answer_label.config(text="Нажата кнопка «Отмена»")
    elif name == "":
        answer_label.config(text="Нажали ОК, но строка пустая")
    else:
        answer_label.config(text=f"Имя: {name}")


tk.Button(root, text="Спросить: удалить?", command=ask_delete).pack(pady=5)
tk.Button(root, text="Спросить: название?", command=ask_name).pack(pady=5)
root.mainloop()


In [ ]:
run_gui("demo_02_questions.py")

### Вопросы для класса

1. Что произойдёт, если в вопросе об удалении нажать «Нет»?
2. Почему `if name is None` не равно `if name == ""`?
3. Если программа получила `True`, должна ли она **обязательно** удалить файл? Нет, мы ещё сами решаем, какое действие выполнить.

**Мини-задание 2:** замените вопрос «Удалить учебную заметку?» на «Сбросить все очки в игре?». Измените вывод при ответе «Да» и «Нет».

---
# Часть 2. Создаём собственное диалоговое окно

## 3. `Tk` и `Toplevel` — главное и дополнительное окно

Представьте, что **основное окно — это тетрадь**, а дополнительное `Toplevel` — страница, открытая для редактирования.

По методичке понадобятся два класса:

- `MainWindow(tk.Tk)` — большое главное окно со списком заметок.
- `TextEditWindow(tk.Toplevel)` — второе окно с текстовым редактором.

А ещё:

- `scrolledtext.ScrolledText` — большое поле, в котором можно писать много строк и прокручивать текст.
- `.grab_set()` — временно переключает управление на окно редактора, чтобы не нажимать на кнопки главного окна, пока редактирование не закончено.
- `.destroy()` — закрывает конкретное окно.

**Преподавателю:** покажите руками, что редактор — отдельное окно, и только после этого разберите код.

## 4. Сначала создадим `TextEditWindow.py`

Сделаем **первую, упрощённую версию**. Пока заметки не записываются на диск: по нажатию «Сохранить» окно показывает введённый текст и закрывается. Это точно соответствует первому этапу методички.

**Три новые вещи:** `class` — чертёж объекта; `__init__` — настройка при создании; `self` — текущий объект (это окно).

In [ ]:
%%writefile tkinter_notes_lesson7/TextEditWindow.py
import tkinter as tk
from tkinter import scrolledtext, messagebox


class TextEditWindow(tk.Toplevel):
    def __init__(self, parent):
        super().__init__(parent)
        self.title("Редактор текста")
        self.geometry("650x450")

        # Многострочное поле с полосой прокрутки
        self.text_area = scrolledtext.ScrolledText(self, wrap=tk.WORD, font=("Arial", 12))
        self.text_area.pack(fill="both", expand=True, padx=10, pady=10)

        self.button = tk.Button(self, text="Сохранить", command=self.save_note)
        self.button.pack(pady=8)

    def save_note(self):
        new_text = self.text_area.get("1.0", "end-1c")
        print(new_text)  # Вывод в консоль — как в первом примере методички
        messagebox.showinfo("Текст получен", f"Вы написали:\n{new_text[:200]}", parent=self)
        self.destroy()


### Как читать код редактора

- `tk.Toplevel` — создаёт **дополнительное окно**, принадлежащее основному.
- `super().__init__(parent)` — разрешает Python выполнить обычное создание дополнительного окна.
- `.pack(fill="both", expand=True)` — поле растягивается вместе с окном.
- `.get("1.0", "end-1c")` — берём текст **с первого символа до самого конца**, без лишнего последнего перевода строки.
- `self.destroy()` — закрывает только редактор; главное окно останется.

**Важно:** при этом **файл ещё не создан**. Мы только научились получать текст из формы.

## 5. Создадим первое главное окно `MainWindow.py`

В главном окне будет **кнопка `+`**. Нажатие вызывает `create_note_card()`, которая создаёт редактор и переключает управление на него через `.grab_set()`.

In [ ]:
%%writefile tkinter_notes_lesson7/MainWindow.py
import tkinter as tk
from TextEditWindow import TextEditWindow


class MainWindow(tk.Tk):
    def __init__(self):
        super().__init__()
        self.__name = "Приложение для заметок"

    def start(self):
        self.title(self.__name)
        self.geometry("800x600")

        self.add_button = tk.Button(
            self, text="+", font=("Arial", 16), command=self.create_note_card
        )
        self.add_button.pack(side="bottom", pady=5)
        self.mainloop()

    def create_note_card(self):
        edit_note = TextEditWindow(self)
        edit_note.grab_set()


## 6. Точка входа — `main.py`

Этот маленький файл — **пульт запуска**. Он создаёт объект главного окна и вызывает его метод `start()`.

Ту же роль `main.py` выполняет в исходной методичке.

In [ ]:
%%writefile tkinter_notes_lesson7/main.py
from MainWindow import MainWindow

main_window = MainWindow()
main_window.start()


In [ ]:
run_gui("main.py")

### Мини-задание 3 — исследуем редактор

Запустите `main.py`, нажмите `+` и попробуйте:

1. Ввести три строки текста.
2. Изменить размер окна редактора.
3. Нажать «Сохранить» и проверить, закрылось ли **только дополнительное** окно.
4. Ещё раз нажать `+` и убедиться, что текст **не сохранился** — пока это нормально!

**Спросите:** чего не хватает, чтобы заметка осталась после закрытия программы? Ответ: сохранения в файл.

**Для преподавателя:** в исходном первом фрагменте `TextEditWindow` встречается `self.note = note`, но параметр `note` ещё не объявлен. На этом этапе мы убрали строку, а позже добавим её вместе с нужным параметром.

---
# Часть 3. Файлы `.note` и класс `Note`

## 7. Файл — память программы

Пока пользователь пишет, текст находится в оперативной памяти. Если закрыть программу, он исчезнет. Чтобы сохранить текст, запишем его **в файл на диске**.

Наши заметки будут иметь расширение **`.note`**. Это обычные текстовые файлы (как `.txt`), просто с другим окончанием имени. Например, `Домашнее задание.note`.

Все файлы мы поместим в папку `tkinter_notes_lesson7/notes` — это небольшое организационное улучшение по сравнению с исходным примером, где `.note` лежат рядом с кодом.

## 8. Учебный опыт: сначала записываем и читаем без окон

Сделаем крошечный пример в отдельном временном каталоге. Он не меняет реальные заметки приложения.

- `.write_text(...)` — записать текст в файл.
- `.read_text(...)` — прочитать текст из файла.
- `encoding="utf-8"` — чтобы русский язык сохранялся правильно.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as folder:
    demo_file = Path(folder) / "Мой пример.note"
    demo_file.write_text("Моя первая заметка!\nНадо повторить Python.", encoding="utf-8")
    result = demo_file.read_text(encoding="utf-8")
    print("Имя файла:", demo_file.name)
    print("Содержимое файла:")
    print(result)

print("После завершения примера временный файл удалён автоматически.")

### Мини-задание 4 — придумай свою заметку

Измените **две строки** в предыдущем примере: название файла и его содержимое. Например, файл `План на выходные.note` и текст из 2–3 строк.

**Вопрос:** зачем мы написали `encoding="utf-8"`? Чтобы текст с русскими буквами не превращался в непонятные символы.

## 9. Класс `Note`: что он умеет?

По методичке класс `Note` управляет **одной** заметкой. У неё есть имя, текст и несколько действий:

| Метод | Что делает |
|---|---|
| `get_filename()` | Возвращает имя файла заметки |
| `save_note(new_text)` | Записывает текст и показывает сообщение |
| `get_note_text()` | Читает текст из файла |
| `delete_note()` | Спрашивает разрешение и удаляет файл |
| `create_note()` | Спрашивает название и создаёт новый файл |

**Простая аналогия:** объект `Note` — это карточка в библиотеке. Он знает, **в каком файле лежит текст**, и умеет читать и изменять этот файл.

`__filename` с двумя подчёркиваниями — внутреннее поле объекта. Для чтения имени мы используем `get_filename()`.

## 10. Что такое `@staticmethod`?

Иногда действие не относится к **уже существующей** заметке. Например, «создать новую заметку» — пока нет конкретного объекта, к которому обращаться.

Поэтому в методичке перед `create_note()` пишется `@staticmethod`. Так можно вызвать `Note.create_note()` **без** команды `Note(...)`.

Попробуем сначала на игрушечном примере, без файлов и окон.

In [ ]:
class GameMessage:
    @staticmethod
    def hello():
        return "Привет, создатель заметок!"

print(GameMessage.hello())

## 11. Создаём полноценный `Note.py`

Ниже — **рабочая версия того же класса, что и в методичке**, с несколькими полезными исправлениями:

- **Отмена** не создаёт файл.
- **Пустое имя** автоматически превращается в `note-...note` с помощью `uuid` (как описано в теории методички; в её коде это условие случайно не выполняется).
- Если файл **уже существует**, программа предупреждает об этом, вместо молчаливой перезаписи.
- Для имён файлов нельзя вводить символы `\ / : * ? " < > |`.
- Русский текст читается/записывается через UTF-8.

В этом фрагменте много строк: для младших учеников разберите сначала только создание, сохранение и удаление; остальные строки объясняйте, когда программа начнёт работать.

In [ ]:
%%writefile tkinter_notes_lesson7/Note.py
import os
import uuid
from pathlib import Path
from tkinter import simpledialog, messagebox

# Все .note-файлы будут находиться в отдельной папке notes.
NOTES_DIR = Path(__file__).resolve().parent / "notes"
NOTES_DIR.mkdir(exist_ok=True)


class Note:
    def __init__(self, filename):
        self.__filename = filename

    def get_filename(self):
        return self.__filename

    def get_note_text(self):
        """Прочитать текст заметки."""
        path = NOTES_DIR / self.__filename
        with open(path, "r", encoding="utf-8") as file:
            return file.read()

    def save_note(self, new_text):
        """Сохранить изменённый текст заметки."""
        path = NOTES_DIR / self.__filename
        with open(path, "w", encoding="utf-8") as file:
            file.write(new_text)
        messagebox.showinfo("Успех", "Заметка сохранена!")

    def delete_note(self):
        """Удалить заметку только после подтверждения пользователя."""
        answer = messagebox.askyesno(
            "Удалить заметку?", f"Точно удалить «{self.__filename}»?"
        )
        if answer:
            os.remove(NOTES_DIR / self.__filename)
            return True
        return False

    @staticmethod
    def create_note():
        """Спросить имя и создать новый пустой файл .note."""
        name = simpledialog.askstring("Новая заметка", "Введите имя заметки:")

        if name is None:  # Пользователь нажал «Отмена»
            return None

        name = name.strip()
        if name == "":
            # Генерируем короткое случайное имя, если поле было пустым.
            name = "note-" + uuid.uuid4().hex[:8]

        if name.lower().endswith(".note"):
            name = name[:-5]

        # Эти символы запрещены в именах файлов во многих системах.
        if not name or name in (".", "..") or any(ch in name for ch in '\\/:*?"<>|'):
            messagebox.showerror("Ошибка", "Недопустимое имя заметки")
            return None

        filename = name + ".note"
        path = NOTES_DIR / filename

        if path.exists():
            messagebox.showwarning("Внимание", "Заметка с таким именем уже есть!")
            return None

        # Режим x создаёт файл, но не перезаписывает существующий.
        try:
            with open(path, "x", encoding="utf-8"):
                pass
        except FileExistsError:
            messagebox.showwarning("Внимание", "Заметка с таким именем уже есть!")
            return None

        messagebox.showinfo("Успех", "Заметка создана!")
        return Note(filename)


### Разбор `Note.py` на простом языке

- `NOTES_DIR` — адрес папки, где живут заметки.
- `with open(path, "w", ...)` — открыть файл для **записи**, записать текст, закрыть файл. Важно: режим `"w"` заменяет прежний текст — именно этого мы и хотим при сохранении изменений.
- `with open(path, "r", ...)` — открыть для **чтения**.
- `os.remove(...)` — удалить файл. Поэтому сначала спрашиваем `askyesno`.
- `uuid.uuid4().hex[:8]` — создаёт короткую случайную последовательность для запасного имени. Это **не идентификатор файловой системы**, а случайный идентификатор, который мы используем в названии файла.
- `return None` — означает, что новая заметка не была создана.

**Задайте вопрос:** зачем проверять `path.exists()`? Чтобы не стереть чужую готовую заметку с таким же названием.

## 12. Маленькая проверка чтения файла через `Note`

Этот тест работает **без графического окна**: создадим специальную учебную заметку в папке `notes`, прочитаем текст через метод класса и удалим лишь этот временный тестовый файл.

In [ ]:
import sys
from pathlib import Path

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from Note import Note, NOTES_DIR

# Используем отдельный временный файл только для проверки.
test_path = NOTES_DIR / "__urok7_proverka__.note"
if test_path.exists():
    print("Тестовый файл уже существует — не перезаписываем его.")
else:
    test_path.write_text("Мои планы: изучить tkinter!", encoding="utf-8")
    test_note = Note(test_path.name)
    print("Имя:", test_note.get_filename())
    print("Текст:", test_note.get_note_text())
    test_path.unlink()  # Удаляем только файл, созданный в этой ячейке.
    print("Проверка завершена.")

### Мини-задание 5 — предскажи результат

Что произойдёт, если в `get_note_text()` заменить режим `"r"` на `"w"`? **Правильный ответ:** файл будет открыт для записи, его содержимое сразу очистится, а `file.read()` вызовет ошибку. Поэтому не меняйте режим случайно.

**Для преподавателя:** покажите пальцем разницу между `"r"`, `"w"` и `"x"` в готовом коде, а не запускайте намеренно опасный пример на настоящих заметках.

---
# Часть 4. Находим все заметки — `Methods`

## 13. Список файлов и расширение `.note`

По методичке создаём класс `Methods` со статическим методом `get_note_list()`.

Он должен:

1. Посмотреть, какие файлы лежат в папке `notes`.
2. Оставить только имена, заканчивающиеся на `.note`.
3. Вернуть список подходящих имён.

Чтобы читать это было проще, начнём с примера обычного списка.

In [ ]:
files = ["кот.note", "кот.jpg", "домашка.note", "main.py"]

for filename in files:
    if filename.endswith(".note"):
        print("Найдена заметка:", filename)

### А теперь то же самое — короткой строкой

`[file for file in files if file.endswith(".note")]` — **генератор списка** (list comprehension). Его можно читать так: «возьми каждый `file` из списка `files`, но только если имя заканчивается на `.note`».

Дети могут сначала пользоваться обычным циклом `for`; короткую запись достаточно узнавать и понимать.

In [ ]:
files = ["кот.note", "кот.jpg", "домашка.note", "main.py"]
notes_only = [file for file in files if file.endswith(".note")]
print(notes_only)

## 14. Создаём `Methods.py`

Реальный метод будет использовать `os.listdir()` для чтения содержимого папки и `.endswith(".note")` — для фильтра. Это тот же способ, что и в методичке; мы только уточняем папку и сортируем имена для удобства.

In [ ]:
%%writefile tkinter_notes_lesson7/Methods.py
import os
from Note import NOTES_DIR


class Methods:
    @staticmethod
    def get_note_list():
        return sorted([
            filename
            for filename in os.listdir(NOTES_DIR)
            if filename.endswith(".note")
            and (NOTES_DIR / filename).is_file()
        ])


In [ ]:
from Methods import Methods

found_notes = Methods.get_note_list()
print("Найдено заметок:", len(found_notes))
print("Имена:", found_notes)

### Мини-задание 6 — фильтры

Как изменить команду в примере со списком, чтобы искать только файлы `.jpg`? Какие имена останутся?

**Ожидаемый ответ:** написать `.endswith(".jpg")`, останется `кот.jpg`.

---
# Часть 5. Соединяем редактор с настоящей заметкой

## 15. Обновляем `TextEditWindow.py`

Раньше окно редактора не знало, **какую заметку** открывает. Теперь передадим ему объект `note`.

Сравните две версии:

- Было: `TextEditWindow(parent)` — просто открывает пустое поле.
- Стало: `TextEditWindow(parent, note)` — открывает **содержимое нужной заметки**.

Добавим `note.get_note_text()` и `note.save_note(new_text)`.

**Исправление ошибки исходного фрагмента:** нужно писать `self.title(note.get_filename())`, а не `self.title = ...`, иначе мы случайно заменим метод установки заголовка строкой.

In [ ]:
%%writefile tkinter_notes_lesson7/TextEditWindow.py
import tkinter as tk
from tkinter import scrolledtext
from Note import Note


class TextEditWindow(tk.Toplevel):
    def __init__(self, parent, note: Note):
        super().__init__(parent)
        self.note = note
        self.geometry("700x500")
        self.title(note.get_filename())

        self.text_area = scrolledtext.ScrolledText(
            self, wrap=tk.WORD, font=("Arial", 12)
        )
        self.text_area.pack(fill="both", expand=True, padx=10, pady=10)
        self.text_area.insert("1.0", note.get_note_text())

        self.button = tk.Button(self, text="Сохранить", command=self.save_note)
        self.button.pack(pady=8)

    def save_note(self):
        new_text = self.text_area.get("1.0", "end-1c")
        self.note.save_note(new_text)
        self.destroy()


### Покажите ученикам путь текста

**Нажали «Редактировать»** → объект `Note` читает текст → редактор показывает его в `ScrolledText` → пользователь меняет текст → нажимает «Сохранить» → `Note.save_note()` записывает файл → появляется сообщение.

**Важно:** если закрыть окно редактора крестиком, не нажав «Сохранить», изменения не запишутся. На сегодняшнем уроке это ожидаемое поведение; предупреждение о несохранённых изменениях — дополнительная задача.

---
# Часть 6. Делаем список карточек в главном окне

## 16. Что такое `Frame`?

`Frame` — это **контейнер**, как коробка, в которую кладут другие элементы интерфейса.

Главное окно будет содержать:

- большой `frame_notes` — место для списка заметок;
- кнопку `+` внизу;
- у каждой заметки — отдельную **карточку**: название, кнопки «Редактировать» и «Удалить».

В методичке также используется список `self.__files`, куда будут складываться **объекты `Note`**.

## 17. Сначала посмотрите на три метода

- `refresh_notes_list()` удаляет старые карточки, очищает список объектов и заново читает файлы `.note`.
- `add_note_to_note_list(note)` добавляет один объект `Note` в список.
- `show_note_card()` проходит по списку и вызывает `add_note_card(note)` для каждого объекта.

Зачем две операции — обновить данные и нарисовать карточки? Это удобно для понимания: **сначала собираем сведения, потом показываем их**.

Ниже — главный итоговый файл. Из-за количества строк сначала покажите детям **действия программы**, а затем разберите его разделами по комментариям.

## 18. Полный класс `MainWindow.py` — все действия из методички

Методы в готовом приложении:

| Метод | Назначение |
|---|---|
| `start` | Настраивает и запускает главное окно |
| `refresh_notes_list` | Читает список файлов, убирает старые карточки |
| `add_note_to_note_list` | Добавляет объект в список |
| `show_note_card` | Рисует карточки всех заметок |
| `add_note_card` | Создаёт одну карточку с кнопками |
| `edit_note` | Открывает окно редактирования |
| `delete_note` | Просит подтверждение удаления через `Note` |
| `create_note_card` | Создаёт новый файл и обновляет список |

**`lambda` в кнопке** означает «при нажатии запусти действие с **этой** заметкой». Мы не вызываем функцию при создании кнопки.

In [ ]:
%%writefile tkinter_notes_lesson7/MainWindow.py
import tkinter as tk
from Methods import Methods
from Note import Note
from TextEditWindow import TextEditWindow


class MainWindow(tk.Tk):
    def __init__(self):
        super().__init__()
        self.add_button = None
        self.frame_notes = None
        self.__files = []
        self.__name = "Приложение для заметок"

    def start(self):
        self.title(self.__name)
        self.geometry("800x600")

        header = tk.Label(self, text="МОИ ЗАМЕТКИ", font=("Arial", 20, "bold"))
        header.pack(pady=12)

        # Общий контейнер для карточек заметок.
        self.frame_notes = tk.Frame(self, bg="#f2f4f8")
        self.frame_notes.pack(padx=12, pady=8, fill="both", expand=True)

        self.add_button = tk.Button(
            self,
            text="+ Добавить заметку",
            font=("Arial", 13),
            command=self.create_note_card
        )
        self.add_button.pack(side="bottom", pady=12)

        self.refresh_notes_list()
        self.show_note_card()
        self.mainloop()

    def refresh_notes_list(self):
        # Убираем старые карточки, чтобы не было повторов.
        for widget in self.frame_notes.winfo_children():
            widget.destroy()

        self.__files.clear()
        for note_name in Methods.get_note_list():
            self.add_note_to_note_list(Note(note_name))

    def add_note_to_note_list(self, note: Note):
        self.__files.append(note)

    def show_note_card(self):
        if not self.__files:
            tk.Label(
                self.frame_notes,
                text="Пока нет заметок. Нажми «+ Добавить заметку»",
                bg="#f2f4f8", font=("Arial", 12)
            ).pack(pady=30)

        for note in self.__files:
            self.add_note_card(note)

    def add_note_card(self, note: Note):
        # Маленькая белая карточка с названием и кнопками.
        card = tk.Frame(self.frame_notes, bg="white", bd=1, relief="solid")
        card.pack(padx=10, pady=5, fill="x")

        title = tk.Label(card, text=note.get_filename(), bg="white", font=("Arial", 12))
        title.pack(side="left", padx=10, pady=12)

        edit_button = tk.Button(
            card, text="Редактировать", command=lambda: self.edit_note(note)
        )
        edit_button.pack(side="right", padx=6)

        delete_button = tk.Button(
            card, text="Удалить", command=lambda: self.delete_note(note)
        )
        delete_button.pack(side="right", padx=6)

    def edit_note(self, note: Note):
        edit_window = TextEditWindow(self, note)
        edit_window.grab_set()

    def delete_note(self, note: Note):
        deleted = note.delete_note()
        if deleted:
            self.refresh_notes_list()
            self.show_note_card()

    def create_note_card(self):
        created = Note.create_note()
        if created is not None:
            self.refresh_notes_list()
            self.show_note_card()


## 19. Запускаем готовое приложение

В `main.py` **ничего менять не нужно** — он по-прежнему создаёт `MainWindow()` и запускает `start()`. Мы заменили содержимое классов, но сохранили их названия.

**Преподавателю:** обязательно закройте старое окно, открытое на этапе 6: та программа уже загрузила **старые версии** классов. Затем запускайте следующую ячейку.

In [ ]:
run_gui("main.py")

### Практическая проверка приложения — обязательно выполнить

**Сценарий 1. Создание заметки**

1. Нажмите `+ Добавить заметку`.
2. Назовите заметку `Покупки`.
3. Убедитесь, что появилась карточка `Покупки.note`.

**Сценарий 2. Редактирование**

1. Нажмите «Редактировать» напротив `Покупки.note`.
2. Введите три строки: `Молоко`, `Хлеб`, `Яблоки`.
3. Нажмите «Сохранить».
4. Снова откройте заметку — все три строки должны сохраниться.

**Сценарий 3. Удаление**

1. Нажмите «Удалить», затем **«Нет»** — заметка должна остаться.
2. Нажмите «Удалить» ещё раз, затем **«Да»** — карточка исчезнет.

**Сценарий 4. Пустое имя и отмена**

1. Нажмите `+`, оставьте имя пустым и нажмите ОК — появится заметка с названием вида `note-a1b2c3d4.note`.
2. Нажмите `+`, затем **«Отмена»** — никакой новой заметки не появится.

**Сценарий 5. Повторное открытие**

Закройте приложение, запустите `main.py` ещё раз и проверьте, что сохранённые заметки остались. Файлы живут в папке `notes`.

## 20. Где хранятся все файлы?

В проекте сейчас есть:

```text
tkinter_notes_lesson7/
├── main.py
├── MainWindow.py
├── TextEditWindow.py
├── Note.py
├── Methods.py
├── notes/
│   ├── Покупки.note
│   └── ...
├── demo_01_messages.py
└── demo_02_questions.py
```

Имена в `notes` зависят от того, какие заметки вы создали. Не удаляйте чужие файлы вручную; используйте для этого кнопку приложения.

In [ ]:
print("Файлы Python в проекте:")
for path in sorted(PROJECT.glob("*.py")):
    print("-", path.name)

print("\nСохранённые заметки:")
for path in sorted((PROJECT / "notes").glob("*.note")):
    print("-", path.name)

---
# Часть 7. Практика и закрепление

## 21. Самостоятельная работа: уровень 1 (для всех)

**Задание А.** Измените заголовок главного окна на «Мои суперзаметки».

**Задание Б.** Поменяйте надпись на кнопке `+ Добавить заметку`, например на «Создать запись».

**Задание В.** Сделайте окно редактора размером `600x400`.

**Задание Г.** Создайте заметку `Мои цели.note`, запишите туда три цели на неделю. Закройте приложение и проверьте, что текст сохранился.

**Подсказка преподавателю:** задания А и Б выполняются в `MainWindow.py`, задание В — в `TextEditWindow.py`, Г — в запущенной программе. После изменения кода нужно заново запустить ячейку `%%writefile` с соответствующим файлом и перезапустить приложение.

## 22. Самостоятельная работа: уровень 2 (кто закончил раньше)

- **Задание Д.** При пустом списке заметок поменяйте надпись «Пока нет заметок...» на свою.
- **Задание Е.** Измените цвет карточки заметки на светло-жёлтый. Подсказка: в `add_note_card` есть `bg="white"` (не забудьте про фон названия).
- **Задание Ж.** Добавьте кнопку «О программе», которая показывает `messagebox.showinfo("О программе", "Это мои заметки на Python")`.
- **Задание З. (со звёздочкой)** После удаления заметки показывайте уведомление «Заметка удалена!».

**Правило:** сначала придумай ожидаемый результат, потом измени 1–3 строки и проверь.

## 23. Дополнение для быстрых учеников: что такое `lambda`?

В нашем коде есть строка:

```python
command=lambda: self.edit_note(note)
```

Если написать `command=self.edit_note(note)`, метод **запустится сразу при создании кнопки**. А нам нужно, чтобы он запускался **позже, при нажатии**.

`lambda` здесь — короткая функция без отдельного имени. Простое сравнение без графики:

In [ ]:
def say_hello():
    print("Привет!")

# say_hello() — ВЫЗОВ прямо сейчас.
# say_hello   — ПЕРЕДАЁМ функцию, чтобы вызвать позже.
# lambda: say_hello() — СОЗДАЁМ маленькую функцию для вызова позже.

later = lambda: say_hello()
print("Функция создана — приветствия ещё нет")
later()

## 24. Проверяем, что файлы с кодом можно прочитать как Python

Это служебная проверка. Она не открывает графические окна и не меняет заметки. Если всё в порядке, появятся строки со словом `OK`.

In [ ]:
import py_compile

for filename in ["main.py", "MainWindow.py", "TextEditWindow.py", "Note.py", "Methods.py"]:
    py_compile.compile(str(PROJECT / filename), doraise=True)
    print("OK:", filename)

## 25. Контрольные вопросы (можно задать устно)

1. Для чего нужны **диалоговые окна**?
2. Чем отличается `Tk()` от `Toplevel()`?
3. Какое окно помогает спросить «Да» или «Нет»?
4. Что возвращает `askstring()`, если нажать «Отмена»?
5. Зачем нужен `ScrolledText`?
6. Что делает `grab_set()`?
7. Для чего нужны `"r"`, `"w"` и `"x"` при открытии файла?
8. Как работает `filename.endswith(".note")`?
9. Почему при удалении нужно спрашивать подтверждение?
10. Зачем нам классы `Note`, `Methods`, `TextEditWindow`, `MainWindow`?

<details>
<summary>Ответы преподавателю — нажмите для раскрытия</summary>

1. Для короткого сообщения, вопроса или ввода данных.
2. `Tk` — главное окно, `Toplevel` — дополнительное.
3. `messagebox.askyesno`.
4. `None`.
5. Для ввода большого текста с прокруткой.
6. Переключает управление на редактор, пока он открыт.
7. Чтение, перезапись, создание нового файла без перезаписи существующего.
8. Проверяет расширение файла.
9. Защита от случайного удаления.
10. Модель заметки, поиск файлов, окно редактора, главное окно со списком.

</details>

## 26. Типичные ошибки и их исправление

| Симптом | Причина и исправление |
|---|---|
| `ModuleNotFoundError: ...` | Запустите сначала все ячейки `%%writefile`; запускайте `main.py` через `run_gui`, которая задаёт правильную папку. |
| `NameError: note is not defined` | В первом примере из источника переменной `note` ещё нет; пользуйтесь исправленным `TextEditWindow.py`. |
| `TypeError: 'str' object is not callable` при `title(...)` | Нельзя писать `self.title = ...`, нужно `self.title(...)`. |
| После изменения кода программа выглядит по-старому | Закройте старое окно и запустите новое: старый процесс не обновится сам. |
| Ничего не появляется при запуске | Tkinter требует графического рабочего стола; локальный Jupyter должен быть запущен на компьютере с GUI. |
| Заметка не сохраняется после крестика | Нужно нажать «Сохранить»; закрытие крестиком не записывает изменения. |
| Русский текст отображается неправильно | Убедитесь, что чтение и запись идут с `encoding="utf-8"`. |
| При создании повторного названия появляется предупреждение | Это защита от случайной перезаписи файла с тем же именем. |

**Не делайте `Restart & Run All` при открытых окнах.** Сначала закройте Tkinter-приложения, затем запускайте блоки последовательно.

## 27. Итоговый чек-лист по методичке

Отметьте каждый пункт после демонстрации:

- [ ] Мы познакомились с видами диалоговых окон.
- [ ] Сделали сообщения `showinfo`, `showwarning`, `showerror`.
- [ ] Проверили ответы `askyesno` и `askstring`.
- [ ] Создали два класса: `MainWindow` и `TextEditWindow`.
- [ ] Создали дополнительное окно `Toplevel` с `ScrolledText`.
- [ ] Использовали `.grab_set()` и `.destroy()`.
- [ ] Запустили приложение из `main.py`.
- [ ] Создали класс `Note` с методами чтения, сохранения, удаления и создания.
- [ ] Использовали `@staticmethod`, `uuid` и расширение `.note`.
- [ ] Создали класс `Methods` с поиском `.note` через `endswith`.
- [ ] Передали объект `Note` в `TextEditWindow`.
- [ ] Добавили `frame_notes` и список `__files` в `MainWindow`.
- [ ] Реализовали `refresh_notes_list`, `add_note_to_note_list`, `show_note_card`.
- [ ] Реализовали `add_note_card` с кнопками редактирования и удаления.
- [ ] Реализовали `edit_note`, `delete_note`, `create_note_card`.
- [ ] Проверили создание, редактирование, удаление с подтверждением и сохранность файла после перезапуска.

**Итог:** не просто красивое окно, а маленькое полноценное приложение с графическим интерфейсом и файлами.

## 28. Как завершить демонстрацию

Закройте все дополнительные окна и главное окно приложения. Эта ячейка может завершить **только те процессы**, которые запускались функцией `run_gui()` в текущем сеансе Jupyter. **Сохранённые заметки не удаляются.**

In [ ]:
for process in open_apps:
    if process.poll() is None:
        process.terminate()
print("Отдельные процессы Tkinter завершены (если ещё работали).")

---
### Методическое примечание

Материал построен на заданиях прикреплённой методички **«PYMiddle. Урок №7. Диалоговые окна в tkinter. Пишем приложение „Заметки“»**. Сохранены требуемые классы, методы, диалоги и последовательность действий. Изменения ради надёжности: исправлены неопределённая переменная в раннем редакторе, вызов `title`, случай с пустым названием; добавлены UTF-8, отказ от перезаписи существующей заметки и отдельная папка данных. Все изменения подробно отмечены выше.